In [8]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

In [9]:
import sys
sys.path.append('../src')

In [10]:
from transformers.transformers_l import DropColumnsTransformer, CityBasedImputer, CityMapTransformer
# from transformers.imputation_by_city import CityBasedImputer


In [11]:
class RollingAverageTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns, window, city_column='city'):
        self.columns = columns
        self.window = window
        self.city_column = city_column
        
    def fit(self, X, y=None):
        # Store column indices if X is a DataFrame
        if hasattr(X, 'columns'):
            self.column_indices_ = {col: i for i, col in enumerate(X.columns) 
                                   if col in self.columns}
            self.feature_names_ = list(X.columns)
        else:
            # If X is already a numpy array, use the column positions directly
            self.column_indices_ = {col: col for col in self.columns 
                                   if isinstance(col, int) and col < X.shape[1]}
        return self
    
    def transform(self, X):
        # Convert to DataFrame if it's a numpy array
        is_numpy = isinstance(X, np.ndarray)
        if is_numpy:
            # If input is numpy array, convert to DataFrame
            X_df = pd.DataFrame(X, columns=self.feature_names_ if hasattr(self, 'feature_names_') else None)
        else:
            X_df = X.copy()
        
        # Create new rolling average columns, grouped by city
        for col, idx in self.column_indices_.items():
            new_col_name = f'{col}_rolling_avg_{self.window}'
            
            # Initialize the new column with NaN values
            X_df[new_col_name] = np.nan
            
            # For each city, compute rolling averages separately
            for city in X_df[self.city_column].unique():
                # Create a city mask
                city_mask = X_df[self.city_column] == city
                
                # Get indices for this city
                city_indices = X_df[city_mask].index
                
                # Compute rolling average for this city only
                city_rolling = X_df.loc[city_mask, col].rolling(
                    window=self.window, min_periods=1).mean()
                
                # Assign the rolling values back to the original DataFrame
                X_df.loc[city_indices, new_col_name] = city_rolling.values
        
        # Return numpy array if input was numpy array
        if is_numpy:
            return X_df.values
        return X_df
    
    def get_feature_names_out(self, input_features=None):
        """Return feature names for output features."""
        if input_features is None and hasattr(self, 'feature_names_'):
            input_features = self.feature_names_
        
        output_features = list(input_features)
        for col in self.columns:
            output_features.append(f'{col}_rolling_avg_{self.window}')
        
        return np.array(output_features)

In [17]:
X = pd.read_csv('../src/data/raw/dengue_features_train.csv')
y = pd.read_csv('../src/data/raw/dengue_labels_train.csv')
y = y[['total_cases']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Define the pipeline
pipeline = Pipeline(steps=[
    ('drop_columns', DropColumnsTransformer(columns_to_drop=['week_start_date', 
                     'reanalysis_sat_prescip_amt_mm',
                     'reanalysis_dew_point_temp_k',
                     'ndvi_nw',
                     'ndvi_se',
                     'reanalysis_air_temp_k',
                     'reanalysis_tdtr_k'])),
    ('city_encoder', CityMapTransformer()),
    ('imputer', CityBasedImputer(city_column='city')),
    ('rolling_avg', RollingAverageTransformer(
        columns=['precipitation_amt_mm','reanalysis_avg_temp_k'], 
        window=3)),
    ('drop_source_columns', DropColumnsTransformer(columns_to_drop=[
         'precipitation_amt_mm', 
         'reanalysis_avg_temp_k'
    ])),
    ('city_onehot', ColumnTransformer([('onehot', OneHotEncoder(sparse_output=False, drop='first'), ['city'])], remainder='passthrough')), 
    ('model', RandomForestRegressor(n_estimators=100, random_state=0))
])


# Fit the pipeline
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

# Calculate the mean absolute error
mean_absolute_error(y_test, y_pred)


/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


12.929623287671232

In [18]:
X_test_competition  = pd.read_csv('../src/data/raw/dengue_features_test.csv')

pipeline.fit(X, y)
predictions = pipeline.predict(X_test_competition)

# round the predictions to the nearest integer
predictions = np.round(predictions).astype(int)

# Create a DataFrame for the predictions as city,year,weekofyear,total_cases
predictions_df = pd.DataFrame({
    'city': X_test_competition['city'],
    'year': X_test_competition['year'],
    'weekofyear': X_test_competition['weekofyear'],
    'total_cases': predictions
})

# Show predictions
print(predictions_df.head())

# Save the predictions to a CSV file
predictions_df.to_csv('../src/data/predictions/baseline_prediction.csv', index=False)


/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


  city  year  weekofyear  total_cases
0   sj  2008          18            4
1   sj  2008          19            5
2   sj  2008          20            4
3   sj  2008          21            6
4   sj  2008          22            7


In [14]:

from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

X = pd.read_csv('../src/data/raw/dengue_features_train.csv')
y = pd.read_csv('../src/data/raw/dengue_labels_train.csv')
y = y[['total_cases']]

#check if ndvi_ne exists in the dataframe
if 'ndvi_ne' in X.columns:
    # Drop the 'ndvi_ne' column
    # # Drop the columns with too many missing values
    X.drop(columns=['week_start_date','reanalysis_sat_precip_amt_mm','reanalysis_dew_point_temp_k','ndvi_nw','ndvi_se'], inplace=True)


# Transform categorical values in the 'city' column to numerical values
X['city'] = X['city'].map({'sj': 0, 'iq': 1})

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Identify columns with missing values in the training set
columns_with_missing = X_train.columns[X_train.isnull().any()]

# Check if there are columns with missing values
if not columns_with_missing.empty:
    # Create a SimpleImputer instance with strategy='mean'
    imputer = SimpleImputer(strategy='mean')

    # Fit the imputer on the training set and transform both training and testing sets
    X_train[columns_with_missing] = imputer.fit_transform(X_train[columns_with_missing])
    X_test[columns_with_missing] = imputer.transform(X_test[columns_with_missing])

model = RandomForestRegressor(n_estimators=100, random_state=0)

# Fit the model
model.fit(X_train, y_train)
# Make predictions
y_pred = model.predict(X_test)
# Calculate the mean absolute error
mean_absolute_error(y_test, y_pred)


/opt/anaconda3/envs/dsr_challenge-env/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


13.27013698630137